# AI Document Assistant — Pipeline Prototyping

Extract -> clean -> chunk -> embed -> store -> retrieve -> generate.
This is the exact pipeline used in `app.py`.

In [1]:
# === Imports ===
from pypdf import PdfReader
import re
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
import chromadb
import ollama

c:\Users\YAMINI SAHU\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# === Extract ===
reader = PdfReader("pwc-global-annual-review-2025.pdf")
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

In [10]:
# === Clean ===
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

cleaned_text = clean_text(full_text)

In [11]:
# === Chunk (sentence-aware, never cuts a sentence mid-word) ===
def chunk_text(text, max_chunk_size=1000):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chunk_size:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

chunks = chunk_text(cleaned_text)
print(f"{len(chunks)} chunks created")

68 chunks created


In [12]:
# === Embed ===
model = SentenceTransformer("all-MiniLM-L6-v2")
all_embeddings = model.encode(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1237.66it/s]


In [13]:
# === Store in ChromaDB ===
client = chromadb.PersistentClient(path="./chroma_db")

try:
    client.delete_collection("pwc-global-annual-review-2025.pdf")
except Exception:
    pass  # collection didn't exist yet — nothing to delete, that's fine

collection = client.create_collection(name="pwc-global-annual-review-2025.pdf")
collection.add(
    documents=chunks,
    embeddings=all_embeddings.tolist(),
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)
print(f"Stored {collection.count()} chunks")

Stored 68 chunks


In [14]:
# === Retrieve ===
query = "What is the purpose of Company report ?"
query_embedding = model.encode(query).tolist()
results = collection.query(query_embeddings=[query_embedding], n_results=3)
print("Retrieved chunks:", len(results['documents'][0]))

Retrieved chunks: 3


In [15]:
# === Generate (final prompt — the one actually used in app.py) ===
def generate_answer(query, retrieved_chunks):
    context = "\n\n".join(retrieved_chunks)
    prompt = f"""You are a helpful AI document assistant. Use the context below to answer the question as accurately as possible.

Context:
{context}

Question: {query}

Instructions for your answer:
- Start with a one-sentence direct answer.
- If there are multiple relevant points, list them as short bullet points.
- Keep the language clear and simple, no unnecessary jargon.
- If the context does not contain enough information, say so clearly instead of guessing.


Answer:"""

    response = ollama.chat(model="llama3.2", messages=[
        {"role": "user", "content": prompt}
    ])
    return response["message"]["content"]

answer = generate_answer(query, results['documents'][0])
print(answer)

The purpose of a Company report is not explicitly stated in the context provided. However, it can be inferred that a Company report is a document or record that contains information about a company's financial performance, governance, and quality management, which is consistent with the context's discussion of the PwC network's quality services and quality management systems.


In [16]:
# === Sanity check: grounding on an unrelated question ===
query_france = "What is the capital of France?"
query_france_embedding = model.encode(query_france).tolist()
results_france = collection.query(query_embeddings=[query_france_embedding], n_results=3)
print(generate_answer(query_france, results_france['documents'][0]))

The question "What is the capital of France?" cannot be answered based on the provided context. The context appears to be related to PwC's annual review, financial reports, and trends in the global economy, AI, and climate change, but it does not contain any information about the capital of France.


## Top-K experiment

Comparing retrieval depth (k = 1, 3, 5) on a set of known questions, to justify
why the app uses `n_results=3` rather than an arbitrary default.

For each question below, note the expected answer from the document yourself,
then read the k=1 / k=3 / k=5 outputs and record whether more context helped,
made no difference, or introduced irrelevant/noisy chunks.

In [17]:
# Evaluation questions based on the PwC Global Annual Review 2025
eval_questions = [
    "What was PwC's global revenue for the year ending 30 June 2025?",
    "How much did PwC invest across its global network in 2025?",
    "How much did PwC invest in expanding and scaling its AI capabilities?",
    "How many people were part of PwC's global network in 2025?",
    "How many countries did PwC operate in according to the Global Annual Review 2025?",
    "What percentage of the Fortune Global 500 does PwC work with?",
    "What were PwC's revenues in the Americas in 2025?",
    "What were PwC's revenues in Europe, Middle East and Africa in 2025?",
    "What were PwC's revenues in Asia Pacific in 2025?",
    "What are PwC's three main lines of service mentioned in the financial performance section?"
]

In [18]:
n_results = 1

In [19]:
n_results = 3

In [20]:
n_results = 5

In [21]:
n_results = 10

In [22]:
def run_topk_comparison(question, k_values=(1, 3, 5, 10)):

    print(f"\n{'='*80}")
    print(f"QUESTION: {question}")
    print(f"{'='*80}")

    q_embedding = model.encode(question).tolist()

    for k in k_values:

        results = collection.query(
            query_embeddings=[q_embedding],
            n_results=k
        )

        retrieved = results["documents"][0]

        # Generate answer using only the retrieved chunks
        answer = generate_answer(question, retrieved)

        print(f"\n--- k={k} ({len(retrieved)} chunks retrieved) ---")
        print(answer)

        print("\nRetrieved chunks:")
        for i, chunk in enumerate(retrieved, 1):
            print(f"\n[Chunk {i}]")
            print(chunk[:500])


for q in eval_questions:
    run_topk_comparison(q)


QUESTION: What was PwC's global revenue for the year ending 30 June 2025?

--- k=1 (1 chunks retrieved) ---
The global revenue for the year ending 30 June 2025 was $56.9 billion.

• This represents an increase of 2.9% in US dollars and 2.7% in local currency compared to the previous financial year.
• The revenue is based on the gross revenues of PwC firms around the world.

Retrieved chunks:

[Chunk 1]
Learn more Learn moreLearn moreLearn more 19 PwCGlobal Annual Review 2025 | Our financial performance Our financial performance Global revenue For the 12 months ending 30 June 2025, PwC firms around the world recorded gross revenues of US$56.9 billion, an increase of 2.9% in US dollars and 2.7% in local currency over the previous financial year’s gross revenues of US$55.3 billion. This is a solid performance in a challenging economic climate and reflects the high quality of the work our 364,000 

--- k=3 (3 chunks retrieved) ---
PwC's global revenue for the year ending 30 June 2025 was 

### Results — run on PwC Global Annual Review 2025

Ran the comparison above on a second, unrelated document (PwC's FY2025 Global
Annual Review, ~59 chunks) across ~8 questions covering direct facts, regional
figures, and one question requiring a specific numeric chunk.

| k | Observation |
|---|---|
| 1 | Correctly answers when the fact is in a single well-matched chunk (e.g. global revenue, country count). Correctly **declines** rather than guessing when the needed chunk isn't retrieved (e.g. "how much did PwC invest across its network" — genuinely not in the 1 retrieved chunk). |
| 3 | Answers most questions correctly. But on one question, the correct chunk wasn't in the top-3, and instead of declining, the model **hallucinated an incorrect figure** ($1.5B instead of the correct $3.1B) — it had just enough surrounding context to fabricate a plausible-sounding but wrong number, despite the grounding instruction in the prompt. |
| 5 | Correctly answered every question tested, including recovering the k=3 failure above once the right chunk was retrieved. |
| 10 | Same answer quality as k=5 on every question — no additional benefit observed, just more tokens in the prompt. |

**Conclusion:** k=3 is not reliably safe — retrieval misses at k=3 don't always
fail loudly (as "not found"); they can fail silently as a confident wrong
answer, which is worse. k=5 was the smallest k that eliminated the observed
failure, and k=10 added no further benefit. **Changed the app's default
`n_results` from 3 to 5** based on this.